# Transform ScientistCloud Terrain TIFFs to IDX

This standalone notebook downloads or reads local ScientistCloud terrain-parameter TIFF files, converts selected parameters into one OpenVisus IDX dataset, validates the result with a static plot, and launches the OpenVisus dashboard.

The default example uses Tennessee at 30 m resolution with five fields: elevation, hillshade, aspect, slope, and plan curvature.

## 1. Install Requirements

Run this cell if this environment has not already installed the packages from `requirements.txt`. GDAL should be installed with Conda before this step.

In [ ]:
# Optional setup cell. Skip it if you already installed requirements.txt.
# %pip install -r requirements.txt

## 2. Prepare the Environment

In [ ]:
from pathlib import Path
from urllib.parse import quote
import os
import subprocess
import sys

import numpy as np
import OpenVisus as ov
import openvisuspy
import requests
from osgeo import gdal, osr
from matplotlib import pyplot as plt
from tqdm import tqdm

print("Environment ready.")

## 3. Configure the Dataset

Change the resolution, state, or fields here. The notebook reuses local TIFFs when present and downloads only missing files by default.

In [ ]:
DATASET_IDS = {
    "10m": "7d796ffa-2489-40b1-b3f4-fa6c0de7b1fe",
    "30m": "2814af5e-0ca7-49fe-ae9b-f5f2e8c2f538",
}

RESOLUTION = "30m"
STATE = "TN"
FIELDS = ["elevation", "hillshade", "aspect", "slope", "plan_curvature"]

# Options: "local", "download", or "local_or_download".
SOURCE_MODE = "local_or_download"

PROJECT_ROOT = Path.cwd()
TIF_ROOT = PROJECT_ROOT / "data" / "tif"
IDX_ROOT = PROJECT_ROOT / "data" / "idx"
LOG_ROOT = PROJECT_ROOT / "logs"

LOCAL_TIF_DIR = TIF_ROOT / RESOLUTION / STATE
IDX_PATH = IDX_ROOT / f"CONUS_{RESOLUTION}_{STATE}_terrain_parameters.idx"
LOG_FILE = LOG_ROOT / "dashboard.log"

PORT = "8989"
ADDRESS = "0.0.0.0"

if RESOLUTION not in DATASET_IDS:
    raise ValueError(f"Unsupported RESOLUTION={RESOLUTION!r}; choose one of {sorted(DATASET_IDS)}")

print(f"Resolution: {RESOLUTION}")
print(f"State: {STATE}")
print(f"Fields: {', '.join(FIELDS)}")
print(f"Local TIFF directory: {LOCAL_TIF_DIR}")
print(f"Output IDX: {IDX_PATH}")

## 4. Build URLs and Resolve Input Files

In [ ]:
SCIENTISTCLOUD_DOWNLOAD = "https://scientistcloud.com/portal/api/public-s3-download.php"


def scientistcloud_key(resolution, state, field):
    return f"utk/conus/{resolution}/{state}/{field}.tif"


def scientistcloud_url(resolution, state, field):
    dataset_id = DATASET_IDS[resolution]
    key = quote(scientistcloud_key(resolution, state, field), safe="")
    return f"{SCIENTISTCLOUD_DOWNLOAD}?k={key}&dataset={dataset_id}"


download_urls = {field: scientistcloud_url(RESOLUTION, STATE, field) for field in FIELDS}

for field, url in download_urls.items():
    print(f"{field}: {url}")

In [ ]:
def download_file(url, output_path, chunk_size=1024 * 1024):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(output_path, "wb") as file_obj:
            progress = tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=output_path.name,
            )
            with progress:
                for chunk in response.iter_content(chunk_size=chunk_size):
                    if chunk:
                        file_obj.write(chunk)
                        progress.update(len(chunk))


def resolve_input_files(fields):
    if SOURCE_MODE not in {"local", "download", "local_or_download"}:
        raise ValueError("SOURCE_MODE must be 'local', 'download', or 'local_or_download'")

    LOCAL_TIF_DIR.mkdir(parents=True, exist_ok=True)
    resolved = {}

    for field in fields:
        path = LOCAL_TIF_DIR / f"{field}.tif"
        should_download = SOURCE_MODE == "download" or (
            SOURCE_MODE == "local_or_download" and not path.exists()
        )

        if path.exists() and not should_download:
            print(f"Using local file: {path}")
        elif should_download:
            print(f"Downloading {field} to {path}")
            download_file(download_urls[field], path)
        else:
            raise FileNotFoundError(
                f"Missing {path}. Set SOURCE_MODE='local_or_download' or 'download' to fetch it."
            )

        resolved[field] = path

    return resolved


input_files = resolve_input_files(FIELDS)
print("Input files are ready.")

## 5. Inspect TIFF Metadata

This section validates that all selected TIFFs use the same raster dimensions and extracts geographic bounds for the IDX physical box.

In [ ]:
GDAL_DTYPE_MAP = {
    "Byte": ("uint8", np.uint8),
    "Int8": ("int8", np.int8),
    "UInt16": ("uint16", np.uint16),
    "Int16": ("int16", np.int16),
    "UInt32": ("uint32", np.uint32),
    "Int32": ("int32", np.int32),
    "UInt64": ("uint64", np.uint64),
    "Int64": ("int64", np.int64),
    "Float32": ("float32", np.float32),
    "Float64": ("float64", np.float64),
}


def open_raster(path):
    dataset = gdal.Open(str(path))
    if dataset is None:
        raise RuntimeError(f"Could not open raster: {path}")
    if dataset.RasterCount < 1:
        raise RuntimeError(f"Raster has no bands: {path}")
    return dataset


def raster_bounds_4326(path):
    dataset = open_raster(path)
    geotransform = dataset.GetGeoTransform()
    source_ref = osr.SpatialReference(wkt=dataset.GetProjection())
    target_ref = osr.SpatialReference()
    target_ref.ImportFromEPSG(4326)

    if hasattr(osr, "OAMS_TRADITIONAL_GIS_ORDER"):
        source_ref.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        target_ref.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

    transform = osr.CoordinateTransformation(source_ref, target_ref)
    width = dataset.RasterXSize
    height = dataset.RasterYSize

    def pixel_to_world(px, py):
        x = geotransform[0] + px * geotransform[1] + py * geotransform[2]
        y = geotransform[3] + px * geotransform[4] + py * geotransform[5]
        lon, lat, _ = transform.TransformPoint(x, y)
        return lon, lat

    corners = [
        pixel_to_world(0, 0),
        pixel_to_world(width, 0),
        pixel_to_world(0, height),
        pixel_to_world(width, height),
    ]
    lons = [lon for lon, _ in corners]
    lats = [lat for _, lat in corners]
    return min(lons), max(lons), min(lats), max(lats)


def raster_metadata(path):
    dataset = open_raster(path)
    band = dataset.GetRasterBand(1)
    gdal_dtype = gdal.GetDataTypeName(band.DataType)
    if gdal_dtype not in GDAL_DTYPE_MAP:
        raise ValueError(f"Unsupported GDAL data type {gdal_dtype!r} in {path}")
    openvisus_dtype, numpy_dtype = GDAL_DTYPE_MAP[gdal_dtype]
    return {
        "width": dataset.RasterXSize,
        "height": dataset.RasterYSize,
        "gdal_dtype": gdal_dtype,
        "openvisus_dtype": openvisus_dtype,
        "numpy_dtype": numpy_dtype,
        "bounds": raster_bounds_4326(path),
    }


metadata = {field: raster_metadata(path) for field, path in input_files.items()}
expected_shape = (metadata[FIELDS[0]]["height"], metadata[FIELDS[0]]["width"])

for field in FIELDS:
    field_shape = (metadata[field]["height"], metadata[field]["width"])
    if field_shape != expected_shape:
        raise ValueError(f"{field} has shape {field_shape}, expected {expected_shape}")
    print(f"{field}: shape={field_shape}, dtype={metadata[field]['openvisus_dtype']}")

lon_min, lon_max, lat_min, lat_max = metadata[FIELDS[0]]["bounds"]
print(f"Longitude range: {lon_min} to {lon_max}")
print(f"Latitude range: {lat_min} to {lat_max}")
print("TIFF metadata is valid.")

## 6. Convert TIFFs to IDX

In [ ]:
def read_single_band(path, numpy_dtype):
    dataset = open_raster(path)
    band = dataset.GetRasterBand(1)
    width = dataset.RasterXSize
    height = dataset.RasterYSize
    buffer = band.ReadRaster(0, 0, width, height, buf_xsize=width, buf_ysize=height, buf_type=band.DataType)
    if buffer is None:
        raise RuntimeError(f"Could not read raster band from {path}")
    array = np.frombuffer(buffer, dtype=numpy_dtype).reshape(height, width)
    return np.flipud(array).copy()


IDX_ROOT.mkdir(parents=True, exist_ok=True)
fields = [ov.Field(field, metadata[field]["openvisus_dtype"]) for field in FIELDS]
height, width = expected_shape

db = ov.CreateIdx(
    url=str(IDX_PATH),
    dims=[width, height],
    fields=fields,
    arco="4mb",
    physic_box=ov.BoxNd.fromString(f"{lon_min} {lon_max} {lat_min} {lat_max}"),
    time=[0, 0, "%00000d/"],
)

for field, db_field in zip(FIELDS, db.getFields()):
    print(f"Writing field: {field}")
    data = read_single_band(input_files[field], metadata[field]["numpy_dtype"])
    db.write(data, field=db_field)

db.compressDataset(["zip"])
print(f"Created IDX file: {IDX_PATH}")

## 7. Validate the IDX Dataset

In [ ]:
def field_name(field):
    if isinstance(field, str):
        return field
    if hasattr(field, "name"):
        return field.name
    if hasattr(field, "getName"):
        return field.getName()
    return str(field)


db = ov.LoadDataset(str(IDX_PATH))
idx_fields = [field_name(field) for field in db.getFields()]
print(f"IDX fields: {idx_fields}")

missing_fields = [field for field in FIELDS if field not in idx_fields]
if missing_fields:
    raise RuntimeError(f"IDX dataset is missing fields: {missing_fields}")

print("Loaded IDX dataset.")

In [ ]:
PREVIEW_MAX_PIXELS = 1_000_000
preview_resolution = db.getMaxResolution()

while preview_resolution > 0:
    preview = db.read(field=FIELDS[0], max_resolution=preview_resolution)
    if preview.size <= PREVIEW_MAX_PIXELS:
        break
    preview_resolution -= 1

print(
    f"Preview resolution: {preview_resolution} "
    f"(max resolution: {db.getMaxResolution()}, shape: {preview.shape})"
)

cmap_instance = plt.get_cmap("inferno")
fig, axes = plt.subplots(len(FIELDS), 1, figsize=(10, 3 * len(FIELDS)))

if len(FIELDS) == 1:
    axes = [axes]

for ax, field in zip(axes, FIELDS):
    data = preview if field == FIELDS[0] else db.read(field=field, max_resolution=preview_resolution)
    image = ax.imshow(
        data,
        cmap=cmap_instance,
        origin="lower",
        extent=(lon_min, lon_max, lat_min, lat_max),
    )
    ax.set_title(f"{field} - {STATE} {RESOLUTION} Resolution")
    ax.set_xlabel("Longitude (Degrees)")
    ax.set_ylabel("Latitude (Degrees)")
    fig.colorbar(
        image,
        ax=ax,
        fraction=0.046 * data.shape[0] / data.shape[1],
        pad=0.04,
    )

plt.subplots_adjust(wspace=0.4, hspace=0.6)
plt.tight_layout()
plt.show()

## 8. Launch the Interactive Dashboard

Run the next two cells to prepare and launch the dashboard. Stop the dashboard cell before launching another dashboard.

In [ ]:
DASHBOARD_APP = PROJECT_ROOT / "dashboard"
if not (DASHBOARD_APP / "main.py").exists():
    raise FileNotFoundError(f"Missing dashboard app: {DASHBOARD_APP}")
LOG_ROOT.mkdir(parents=True, exist_ok=True)


def launch_dashboard():
    cmd = [
        sys.executable,
        "-m",
        "panel",
        "serve",
        str(DASHBOARD_APP),
        "--log-file",
        str(LOG_FILE),
        "--dev",
        "--allow-websocket-origin=*",
        f"--address={ADDRESS}",
        f"--port={PORT}",
        "--args",
        str(IDX_PATH),
    ]
    subprocess.run(cmd, check=True)


print(f"Dashboard app path: {DASHBOARD_APP}")
print(f"Log file: {LOG_FILE}")
DASHBOARD_ROUTE = DASHBOARD_APP.name
print(f"Visit http://localhost:{PORT}/{DASHBOARD_ROUTE} to explore the dashboard.")

In [ ]:
%%capture dashboard_output
# Stop this cell before running another visualization dashboard.
launch_dashboard()

If the dashboard cell exits immediately, run the diagnostic cell below to show the captured startup output.

In [ ]:
try:
    dashboard_output.show()
except NameError:
    print("Run the dashboard cell first.")

## Notes

- The 10 m files can be many gigabytes per field.
- To use local data only, place TIFFs under `data/tif/<resolution>/<state>/` and set `SOURCE_MODE = "local"`.
- The source field names are preserved in the IDX dataset so the dashboard field selector matches the ScientistCloud terrain parameter names.